# NimbusPay ticket triage — LoRA fine-tuning (Parts B–E)

**Stack:** Hugging Face Transformers + PEFT + bitsandbytes with a small custom training loop in `pipeline.py`, so every setting the brief asks about (chat template, loss mask, sequence length, memory) is visible.

**Base model:** `Qwen/Qwen2.5-3B-Instruct`: under the 4B limit, ungated, with a real `system` role in its chat template.

**Runs on Kaggle (GPU T4 x2, Internet on) or Colab (T4).** On Kaggle, attach the private `candidate_pack` dataset; on Colab, upload the folder to `MyDrive/`.

**Time limit:** the final model and test predictions are the protected path. Experiments share the second GPU and stop starting after `START_LIMIT_MIN`. With one GPU the jobs run one after another.

## Run record: what was actually executed

Colab's T4 was unavailable and Kaggle sessions would not start, so the final model, test predictions and experiments ran on a laptop RTX 5070 (8 GB). The cells in this section read those saved results and show real output. **The cells after this section target a T4 and were not executed on one, so they have no output.** The laptop run used `max_len` 1500 and `token_budget` 1500 to fit 8 GB, which skipped 32 long rows.

Log of the laptop run: when each job started and ended, how many rows it used, and the training loss per epoch.

In [1]:
print(open("runs/local_run.log", encoding="utf-8").read())

#### final start 20:53:16
base params 3.40B | trainable 29.9M
rows 1338 | usable 1306 | skipped 32
epoch 1 step 50/380 loss 0.4348 elapsed 1.4m
epoch 1 step 100/380 loss 0.2303 elapsed 2.5m
epoch 1 step 150/380 loss 0.1574 elapsed 3.7m
== epoch 1 mean loss 0.1260
epoch 2 step 200/380 loss 0.0020 elapsed 5.1m
epoch 2 step 250/380 loss 0.0040 elapsed 6.5m
epoch 2 step 300/380 loss 0.0034 elapsed 7.9m
epoch 2 step 350/380 loss 0.0029 elapsed 9.4m
== epoch 2 mean loss 0.0027
DONE final
#### final end 21:07:13 exit 0
#### test start 21:07:13
DONE shard 0/1
#### test end 21:50:55 exit 0
#### FINAL READY
#### exp_base not started: past cutoff
#### exp_raw not started: past cutoff
#### exp_rank8 not started: past cutoff
#### exp_aug not started: past cutoff
#### baseline_base not started: past cutoff
#### baseline_rules not started: past cutoff
#### exp_16bit not started: past cutoff
#### ALL DONE 2026-11-09 21:50:55.054705
#### exp_base start 21:51:16
base params 3.40B | trainable 29.9M
rows 

Results table, read from the metrics each run saved. Experiments use 200 training rows, 2 epochs and the first 80 dev tickets, so small differences mean little. The final row uses all usable clean rows and all 200 dev tickets.

In [2]:
import json, os
runs = ["baseline_base", "baseline_rules", "exp_base", "exp_raw", "exp_4bit_500", "exp_16bit", "exp_rank8", "exp_aug", "final"]
print(f"{'run':16}{'exact':>8}{'fields':>8}{'JSON ok':>9}{'peak GB':>9}{'train min':>11}{'dev n':>7}")
for n in runs:
    f = f"runs/{n}/metrics.json"
    if not os.path.exists(f):
        print(f"{n:16}did not complete"); continue
    m = json.load(open(f))
    print(f"{n:16}{m['exact_match']:8.1%}{m['mean_field_acc']:8.1%}{m['json_valid']:9.1%}{str(m.get('peak_gpu_gb','-')):>9}{str(m.get('train_minutes','-')):>11}{m['config']['limit'] or 200:7d}")

run                exact  fields  JSON ok  peak GB  train min  dev n
baseline_base       0.0%    0.0%    78.8%        -          -     80
baseline_rules      2.5%   20.8%    28.7%        -          -     80
exp_base           41.2%   86.9%   100.0%     5.46       1.43     80
exp_raw         did not complete
exp_4bit_500       51.2%   89.7%   100.0%     4.42       1.69     80
exp_16bit          77.5%   95.0%   100.0%     7.02       1.65     80
exp_rank8          32.5%   84.7%   100.0%     6.28       1.43     80
exp_aug         did not complete
final              93.0%   98.8%   100.0%     6.45      10.29    200


Part E on the final model: scores by category and field, then every failed dev ticket with the fields it got wrong. `data/dev.jsonl` comes from the assignment pack.

In [3]:
import json
from score import score, load, KEYS
gold, pred = load("data/dev.jsonl"), load("runs/final/dev_predictions.jsonl")
res = score(gold, pred)
for n in sorted(res, key=lambda x: (x != "all", x)):
    print(f"{n:26}{res[n]['n']:5d}{res[n]['exact_match']:8.1%}{res[n]['mean_field_acc']:8.1%}")
print({k: round(v, 3) for k, v in sorted(res["all"]["fields"].items(), key=lambda x: x[1])})
pm = {p["id"]: p["output"] for p in pred}
for g in gold:
    w = json.loads(g["messages"][-1]["content"]); o = json.loads(pm[g["id"]])
    wrong = [k for k in KEYS if k not in o or type(o[k]) is not type(w[k]) or o[k] != w[k]]
    if wrong: print(g["id"], {k: (w[k], o.get(k)) for k in wrong})

all                         200   93.0%   98.8%
category=account_access      19   89.5%   98.7%
category=fraud               15   86.7%   97.5%
category=kyc                 24  100.0%  100.0%
category=offers              18  100.0%  100.0%
category=other               16  100.0%  100.0%
category=payment_failed      53   90.6%   98.3%
category=refund              55   90.9%   98.2%
{'priority': 0.955, 'needs_human': 0.97, 'txn_date': 0.985, 'category': 0.99, 'amount_paise': 1.0, 'txn_id': 1.0, 'channel': 1.0, 'language': 1.0}
dv-00006 {'txn_date': ('2026-06-30', '2026-07-01')}
dv-00034 {'priority': ('P2', 'P1'), 'needs_human': (False, True)}
dv-00037 {'priority': ('P1', 'P2'), 'needs_human': (True, False)}
dv-00055 {'priority': ('P2', 'P3')}
dv-00056 {'priority': ('P2', 'P1'), 'needs_human': (False, True)}
dv-00083 {'priority': ('P2', 'P3')}
dv-00098 {'priority': ('P2', 'P1'), 'needs_human': (False, True)}
dv-00113 {'needs_human': (True, False)}
dv-00114 {'txn_date': ('2026-08-30', None

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2

In [ ]:
import os, sys, json, re, glob, shutil, subprocess, threading, queue, time, zipfile, collections as C
import concurrent.futures as F

KAGGLE = os.path.exists("/kaggle/input")
if KAGGLE:
    found = glob.glob("/kaggle/input/**/SCHEMA.md", recursive=True)
    assert found, "Attach the candidate_pack dataset: SCHEMA.md was not found under /kaggle/input"
    PACK = "/kaggle/working/candidate_pack"
    if not os.path.exists(PACK):
        shutil.copytree(os.path.dirname(found[0]), PACK, ignore=shutil.ignore_patterns("runs", "__pycache__", ".ipynb_checkpoints"))
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PACK = "/content/drive/MyDrive/candidate_pack"      # change this if you uploaded it elsewhere
os.chdir(PACK)
need = ["SCHEMA.md", "score.py", "triage_rules.py", "clean_data.py", "augment.py", "pipeline.py",
        "data/train.jsonl", "data/dev.jsonl", "data/test_inputs.jsonl"]
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"Missing from {PACK}: {missing}"
print("All files found in", PACK)

## 1. Data: rebuild the cleaned training set

The full audit is in `01_data_audit.ipynb`. This reproduces `data/train_clean.jsonl` and `cleaning_log.csv` from the raw file. Augmentation (Part D, own idea) uses only `train_clean.jsonl` and `SCHEMA.md`.

In [ ]:
!python clean_data.py
!python augment.py | head -1

## 2. Pipeline checks: sequence length, chat template, loss mask

**Sequence length is chosen from evidence:** token counts over the training data, including augmented rows. A row longer than `max_len` is skipped rather than truncated, because truncating would cut off the answer.

In [ ]:
import statistics as S
from transformers import AutoTokenizer
import pipeline as P, torch

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
rows = P.training_rows("aug")
lens = sorted(len(P.encode(tok, r, 10**9)["input_ids"]) for r in rows)
q = lambda p: lens[int(p * (len(lens) - 1))]
print(f"rows {len(lens)} | median {q(.5)} | p90 {q(.9)} | p99 {q(.99)} | max {lens[-1]}")
print("rows over max_len 2300:", sum(l > 2300 for l in lens), "| rows over 512:", sum(l > 512 for l in lens))
answers = [len(tok(r["messages"][2]["content"], add_special_tokens=False).input_ids) for r in rows]
print("longest answer in tokens:", max(answers), "(the inference limit is 200 new tokens)")

In [ ]:
ex = P.encode(tok, rows[0], 2300)
n_masked = sum(l == -100 for l in ex["labels"])
print("===== model input =====\n" + tok.decode(ex["input_ids"]))
print("===== tokens the loss is computed on =====\n" + tok.decode([l for l in ex["labels"] if l != -100]))
print(f"\n{n_masked} prompt tokens masked with -100; {len(ex['labels']) - n_masked} answer tokens trained")
assert tok.decode([ex["labels"][-1]]) == "<|im_end|>", "the model must learn to stop"

## 3. Main settings and why

| Setting | Value | Reason |
|---|---|---|
| Quantization | 4-bit NF4, double quant (QLoRA) | Fits a T4 with room for long email threads; compared with 16-bit in an experiment |
| Rank / alpha | 16 / 32 | About 30M trainable, under the 50M cap; compared with rank 8 in an experiment |
| Target modules | all attention and MLP linear layers | The task needs reasoning over rules, not only style |
| Learning rate | 2e-4, linear decay, 5% warmup | Standard LoRA range |
| Final training | cleaned data, 2 epochs | Fits the available GPU time. Chosen in advance of the experiment results |
| Max sequence length | 2300 | From the token counts above. A lower cap would discard the long quoted threads that teach the model to ignore quoted dates |
| Batching | up to 8, capped by `token_budget` 2400 | Long threads make smaller batches, which bounds memory |
| Gradient checkpointing | on | Trades speed for memory on long rows |

## 4. Run helpers

In [ ]:
RUN_EXPERIMENTS = True
RUN_FINAL = True
START_LIMIT_MIN = 40      # no new experiment starts after this many minutes
KILL_LIMIT_MIN = 60       # an experiment still running at this point is stopped; the final model is never stopped
NGPU = max(1, torch.cuda.device_count())
print("GPUs:", NGPU)

def launch(label, args, gpu=0, timeout=None):
    p = subprocess.Popen([sys.executable, "-u", "pipeline.py", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
    timer = threading.Timer(timeout, p.kill) if timeout else None
    if timer: timer.start()
    for line in p.stdout:
        if not re.search(r"[Ww]arning|Loading weights|^\s*$", line): print(f"[{label} gpu{gpu}] {line}", end="", flush=True)
    code = p.wait()
    if timer: timer.cancel()
    print(f"[{label}] {'finished' if code == 0 else f'FAILED or stopped, exit code {code}'}", flush=True)
    return code == 0

def run(name, *args, gpu=0, timeout=None):
    if os.path.exists(f"runs/{name}/metrics.json"):
        print(f"[skip] {name}: results already saved", flush=True); return True
    return launch(name, ["--name", name, *args], gpu, timeout)

def show_table():
    print(f"| run | what changed | dev exact | dev field acc | JSON valid | peak GPU GB | train min |\n|---|---|---|---|---|---|---|")
    for name, what in TABLE:
        f = f"runs/{name}/metrics.json"
        if not os.path.exists(f): print(f"| {name} | {what} | not run |  |  |  |  |"); continue
        m = json.load(open(f))
        print(f"| {name} | {what} | {m['exact_match']:.1%} | {m['mean_field_acc']:.1%} | {m['json_valid']:.1%} | {m.get('peak_gpu_gb','-')} | {m.get('train_minutes','-')} |")

## 5. Parts B and D, and the final model

**Baselines (Part B):** the base model with no training, with the fixed prompt and with the SCHEMA rules in the prompt (allowed only as a reference).

**Experiments (Part D), one change at a time from `exp_base`:** cleaned data, 4-bit, rank 16, lr 2e-4. The `raw` run still removes rows that appear in dev. Every experiment uses the same reduced budget: a fixed random sample of 200 training rows (the own-idea run adds augmented versions of those rows), 2 epochs, batched dev scoring. The comparison between runs is fair, but these scores are lower than a full run would reach and are not final-model scores.

**Final model:** all cleaned rows, 2 epochs. Its dev score comes from the saved adapter reloaded from disk, using batched scoring to save time. **Test predictions use the plain call: one ticket at a time, greedy, `max_new_tokens=200`, on the saved adapter**, split across the GPUs.

In [ ]:
SAME = ["--train_limit", "200", "--epochs", "2", "--eval_batch", "16"]    # identical for every experiment
FINAL_ARGS = ["--epochs", "2", "--eval_batch", "16"]
TABLE = [("baseline_base", "no fine-tuning, fixed prompt"), ("baseline_rules", "no fine-tuning, rules in prompt"),
         ("exp_base", "reference: clean, 4-bit, rank 16 (200 rows, 2 epochs)"),
         ("exp_raw", "1. raw data instead of clean"),
         ("exp_16bit", "2. 16-bit LoRA instead of 4-bit QLoRA"),
         ("exp_rank8", "3. rank 8 instead of 16"),
         ("exp_aug", "4. own idea: SCHEMA-based augmentation"),
         ("final", "final: all clean rows, 2 epochs")]
JOBS = [("exp_base", SAME), ("exp_raw", [*SAME, "--data", "raw"]), ("exp_16bit", [*SAME, "--bits", "16"]),
        ("exp_rank8", [*SAME, "--rank", "8"]), ("exp_aug", [*SAME, "--data", "aug"]),
        ("baseline_base", ["--mode", "base", "--eval_batch", "16"]), ("baseline_rules", ["--mode", "rules", "--eval_batch", "16"])]

T0 = time.time()
elapsed = lambda: (time.time() - T0) / 60
os.makedirs("runs/final", exist_ok=True)
exp_jobs, shards, final_ready, final_failed = queue.Queue(), queue.Queue(), threading.Event(), []
if RUN_EXPERIMENTS:
    for job in JOBS: exp_jobs.put(job)

def next_item(q):
    try: return q.get_nowait()
    except queue.Empty: return None

def worker(gpu):
    if gpu == 0 and RUN_FINAL:
        if not run("final", *FINAL_ARGS, gpu=gpu): final_failed.append("training")
        else:
            for i in range(NGPU): shards.put(i)
        final_ready.set()
    while True:
        if final_ready.is_set() and not final_failed:
            i = next_item(shards)
            if i is not None:
                if not launch(f"test{i}", ["--name", "final", "--mode", "reload", "--test_only", "--test_shard", f"{i}/{NGPU}"], gpu):
                    final_failed.append(f"test shard {i}")
                continue
        job = next_item(exp_jobs)
        if job is None:
            if RUN_FINAL and not final_ready.is_set():
                final_ready.wait(); continue
            return
        name, args = job
        if elapsed() > START_LIMIT_MIN:
            print(f"[not started] {name}: past the {START_LIMIT_MIN}-minute limit", flush=True); continue
        run(name, *args, gpu=gpu, timeout=max(60, (KILL_LIMIT_MIN - elapsed()) * 60))

with F.ThreadPoolExecutor(NGPU) as pool:
    for f in [pool.submit(worker, g) for g in range(NGPU)]: f.result()
print(f"all jobs finished in {elapsed():.1f} min")
assert not final_failed, f"final model steps failed: {final_failed}"
show_table()

## 6. Final deliverables

In [ ]:
test = [json.loads(l) for l in open("data/test_inputs.jsonl", encoding="utf-8")]
by_id = {}
for f in glob.glob("runs/final/predictions_test.part*.jsonl"):
    for l in open(f, encoding="utf-8"):
        x = json.loads(l); by_id[x["id"]] = x
assert len(by_id) == len(test), (len(by_id), len(test))
with open("predictions_test.jsonl", "w", encoding="utf-8") as f:
    for t in test: f.write(json.dumps(by_id[t["id"]], ensure_ascii=False) + "\n")

shutil.rmtree("adapter", ignore_errors=True)
shutil.copytree("runs/final/adapter", "adapter")
for d in glob.glob("runs/*/adapter"):
    if os.sep + "final" + os.sep not in d: shutil.rmtree(d)      # experiment adapters are not needed; their metrics stay

def ok(s):
    try: return isinstance(json.loads(s), dict)
    except Exception: return False
preds = [by_id[t["id"]] for t in test]
m = json.load(open("runs/final/metrics.json"))
print(f"dev (saved adapter, batched scoring): exact {m['exact_match']:.1%}, field acc {m['mean_field_acc']:.1%}, trainable {m['trainable']/1e6:.1f}M, base {m['base_params']/1e9:.2f}B, peak {m['peak_gpu_gb']} GB, train {m['train_minutes']} min")
print(len(preds), "test predictions | valid JSON:", sum(ok(p["output"]) for p in preds))
print({f: f"{os.path.getsize('adapter/'+f)/1e6:.1f} MB" for f in os.listdir("adapter")})

out = ("/kaggle/working/" if KAGGLE else "") + "submission_bundle.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [*glob.glob("adapter/*"), "predictions_test.jsonl", "cleaning_log.csv", *glob.glob("runs/*/metrics*.json"), "runs/final/dev_predictions.jsonl"]:
        z.write(f)
print("download this file:", out)

## 7. Part E — error analysis on dev (final model)

In [ ]:
from score import score, load, KEYS
gold = load("data/dev.jsonl")
pred = load("runs/final/dev_predictions.jsonl")
res = score(gold, pred)
print(f"{'group':26}{'n':>5}{'exact':>8}{'fields':>8}")
for name in sorted(res, key=lambda x: (x != "all", x)):
    print(f"{name:26}{res[name]['n']:5d}{res[name]['exact_match']:8.1%}{res[name]['mean_field_acc']:8.1%}")
print("\nper-field accuracy:")
for k, v in sorted(res["all"]["fields"].items(), key=lambda x: x[1]): print(f"  {k:14}{v:7.1%}")

In [ ]:
pmap = {p["id"]: p["output"] for p in pred}
failures, by_field = [], C.Counter()
for g in gold:
    want = json.loads(g["messages"][-1]["content"])
    try: got = json.loads(pmap[g["id"]])
    except Exception: got = None
    wrong = [k for k in KEYS if got is None or k not in got or type(got[k]) is not type(want[k]) or got[k] != want[k]]
    if wrong:
        failures.append((g, want, got, wrong)); by_field.update(wrong)
print(len(failures), "failures of", len(gold), "| wrong by field:", dict(by_field.most_common()), "\n")
for g, want, got, wrong in failures[:15]:
    text = g["messages"][1]["content"].split("\n> On")[0]
    print("=" * 90, f"\n{g['id']}  wrong: {wrong}\n{text[:600]}")
    for k in wrong: print(f"   {k}: expected {want[k]!r}, got {None if got is None else got.get(k)!r}")